# In-House Layout Clustering Pipeline

**목적**: Layout feature 126개 × 280,000 rows 데이터에서  
- CD (nm) 분산(sigma)을 최소화하는 클러스터 도출  
- SHAP 기반 feature 중요도 분석 및 선택  
- Cost function: `α × mean(σ_per_cluster) + β × max(σ_per_cluster)` 최소화  
- 제약 조건: 클러스터당 최소 샘플 수 ≥ MIN_CLUSTER_COUNT

**알고리즘 전략**:
1. XGBoost로 CD 예측 모델 학습 → SHAP feature 중요도 추출  
2. 중요 feature 선택 후 Decision Tree Clustering (Recursive Partitioning)  
3. 소규모 클러스터 병합(KNN) → 제약 조건 충족  
4. 하이퍼파라미터 grid search로 cost function 최소화  
5. K-Means 비교 기준선 제공

## 0. 설정 (Configuration)

In [ ]:
# ============================================================
#  GLOBAL CONFIG — 이 셀만 수정하면 전체 파이프라인 제어 가능
# ============================================================

# 데이터 경로 설정
DATA_PATH = None          # None이면 합성 데이터 자동 생성 (테스트용)
# DATA_PATH = 'data/layout_features.csv'   # 실제 데이터 경로

CD_COLUMN = 'CD_nm'       # 타겟 컬럼명 (CD in nm)
N_ROWS    = 280_000       # 합성 데이터 행 수
N_FEAT    = 126           # 합성 데이터 feature 수 (CD 제외)

# ── SHAP feature 선택 ────────────────────────────────────────
SHAP_TOP_K         = 20   # SHAP 상위 K개 feature 사용
SHAP_SAMPLE_N      = 10_000  # SHAP 계산용 샘플 수 (속도 vs 정확도)
XGB_N_ESTIMATORS   = 200
XGB_MAX_DEPTH      = 6

# ── 클러스터링 제약 ─────────────────────────────────────────
MIN_CLUSTER_COUNT  = 100  # 클러스터 최소 샘플 수 (핵심 제약)

# ── Cost function 가중치 ─────────────────────────────────────
ALPHA = 1.0   # mean(σ) 가중치
BETA  = 0.5   # max(σ) 가중치

# ── Decision Tree 하이퍼파라미터 탐색 범위 ───────────────────
DT_MAX_LEAF_RANGE  = list(range(5, 81, 5))   # [5, 10, ..., 80]

# ── K-Means 하이퍼파라미터 탐색 범위 ────────────────────────
KMEANS_K_RANGE     = list(range(5, 51, 5))   # [5, 10, ..., 50]

# ── 재현성 ──────────────────────────────────────────────────
RANDOM_STATE = 42

# ── 출력 경로 ────────────────────────────────────────────────
OUTPUT_DIR = 'output'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')
print(f'  CD column       : {CD_COLUMN}')
print(f'  SHAP top-K      : {SHAP_TOP_K}')
print(f'  Min cluster cnt : {MIN_CLUSTER_COUNT}')
print(f'  Cost weights    : α={ALPHA} (mean σ), β={BETA} (max σ)')

## 1. 라이브러리 임포트

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

import xgboost as xgb
import shap

import joblib
import json
from pathlib import Path
from collections import Counter

shap.initjs()
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 2. 데이터 로딩 & EDA

In [ ]:
# ── 2-1. 데이터 로딩 ─────────────────────────────────────────
def generate_synthetic_data(n_rows: int, n_feat: int, cd_col: str, seed: int) -> pd.DataFrame:
    """현실적인 합성 layout feature 데이터 생성.
    
    내부적으로 3개의 잠재 클러스터 그룹을 심어두어
    클러스터링 알고리즘 검증에 사용 가능.
    """
    rng = np.random.default_rng(seed)

    # 잠재 그룹 (ground truth)
    group = rng.choice([0, 1, 2, 3, 4], size=n_rows, p=[0.35, 0.25, 0.20, 0.12, 0.08])

    # Group별 특성이 다른 feature 행렬
    X = rng.standard_normal((n_rows, n_feat)).astype(np.float32)
    for g in range(5):
        mask = group == g
        shift = rng.uniform(-2, 2, n_feat)
        scale = rng.uniform(0.5, 1.5, n_feat)
        X[mask] = X[mask] * scale + shift

    # 일부 feature에 결측치 주입 (현실 반영)
    missing_mask = rng.random((n_rows, n_feat)) < 0.01
    X[missing_mask] = np.nan

    feat_names = [f'feat_{i:03d}' for i in range(n_feat)]
    df = pd.DataFrame(X, columns=feat_names)

    # CD = 선형조합 + group offset + noise
    coef = rng.uniform(-1, 1, n_feat)
    cd_base = df.fillna(0).values @ coef * 0.3
    group_offset = np.array([50, 70, 90, 110, 130])[group]
    noise_std = np.array([3, 5, 4, 6, 8])[group]
    cd = cd_base + group_offset + rng.normal(0, noise_std, n_rows)
    df[cd_col] = cd.astype(np.float32)
    df['_true_group'] = group  # 검증용 (실제 데이터엔 없음)

    return df


if DATA_PATH is None:
    print(f'합성 데이터 생성 중 ({N_ROWS:,} rows × {N_FEAT} feats)...')
    df_raw = generate_synthetic_data(N_ROWS, N_FEAT, CD_COLUMN, RANDOM_STATE)
    print('합성 데이터 생성 완료 (검증용 _true_group 포함)')
else:
    print(f'데이터 로딩: {DATA_PATH}')
    df_raw = pd.read_csv(DATA_PATH)
    print(f'로드 완료: {df_raw.shape}')

print(f'\nShape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# ── 2-2. 기본 EDA ─────────────────────────────────────────────
feat_cols = [c for c in df_raw.columns if c not in [CD_COLUMN, '_true_group']]
print(f'Feature columns : {len(feat_cols)}')
print(f'Target column   : {CD_COLUMN}')
print(f'\n── CD (nm) 기술 통계 ──')
print(df_raw[CD_COLUMN].describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# CD 분포
axes[0].hist(df_raw[CD_COLUMN].dropna(), bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('CD (nm) Distribution')
axes[0].set_xlabel('CD (nm)'); axes[0].set_ylabel('Count')

# 결측치 비율
missing_pct = df_raw[feat_cols].isnull().mean().sort_values(ascending=False)
top_missing = missing_pct[missing_pct > 0].head(20)
if len(top_missing):
    axes[1].barh(top_missing.index[::-1], top_missing.values[::-1], color='salmon')
    axes[1].set_title('Top 20 Missing Rate (features)')
    axes[1].set_xlabel('Missing Ratio')
else:
    axes[1].text(0.5, 0.5, 'No Missing Values', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Missing Values')

# Feature 분산 분포
feat_var = df_raw[feat_cols].var().sort_values()
axes[2].hist(feat_var, bins=50, color='mediumseagreen', edgecolor='none')
axes[2].set_title('Feature Variance Distribution')
axes[2].set_xlabel('Variance'); axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2-3. 저분산 feature 제거 ──────────────────────────────────
VAR_THRESHOLD = 1e-6
low_var_cols = feat_var[feat_var < VAR_THRESHOLD].index.tolist()
print(f'저분산 feature 제거: {len(low_var_cols)}개 ({VAR_THRESHOLD} 이하)')

feat_cols_clean = [c for c in feat_cols if c not in low_var_cols]
print(f'남은 feature 수: {len(feat_cols_clean)}')

## 3. 전처리 (Preprocessing)

In [ ]:
# ── 3-1. 결측치 처리 & 스케일링 ──────────────────────────────

# CD가 결측인 행 제거
df_clean = df_raw.dropna(subset=[CD_COLUMN]).copy()
removed = len(df_raw) - len(df_clean)
print(f'CD 결측 행 제거: {removed:,}개 → 남은 행: {len(df_clean):,}')

# Feature: 중앙값 imputation (결측치가 1% 이하로 가정)
X_df = df_clean[feat_cols_clean].copy()
col_medians = X_df.median()
X_df = X_df.fillna(col_medians)

y = df_clean[CD_COLUMN].values.astype(np.float32)

# RobustScaler: outlier에 덜 민감
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=feat_cols_clean, index=X_df.index)

print(f'전처리 완료: X={X_scaled.shape}, y={y.shape}')
print(f'  y 범위: [{y.min():.2f}, {y.max():.2f}] nm')
print(f'  y σ (전체): {y.std():.3f} nm')

## 4. SHAP 기반 Feature Engineering

In [ ]:
# ── 4-1. XGBoost CD 예측 모델 학습 ───────────────────────────
print('XGBoost CD 예측 모델 학습 중...')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE
)

xgb_model = xgb.XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
    eval_metric='rmse',
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

y_pred_val = xgb_model.predict(X_val)
rmse = np.sqrt(np.mean((y_pred_val - y_val) ** 2))
r2 = 1 - np.sum((y_pred_val - y_val) ** 2) / np.sum((y_val - y_val.mean()) ** 2)

print(f'\nXGBoost 학습 완료:')
print(f'  Validation RMSE : {rmse:.3f} nm')
print(f'  Validation R²   : {r2:.4f}')
joblib.dump(xgb_model, f'{OUTPUT_DIR}/xgb_cd_model.pkl')

In [ ]:
# ── 4-2. SHAP 값 계산 ─────────────────────────────────────────
print(f'SHAP 계산 중 (샘플 {SHAP_SAMPLE_N:,}개)...')

# 속도를 위해 서브샘플 사용
shap_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_scaled), size=min(SHAP_SAMPLE_N, len(X_scaled)), replace=False
)
X_shap = X_scaled[shap_idx]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

# Feature별 mean |SHAP| 계산
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feat_cols_clean)\
                    .sort_values(ascending=False)

print('SHAP 계산 완료.')
print(f'\n상위 {SHAP_TOP_K}개 feature:')
print(shap_importance.head(SHAP_TOP_K).to_string())

In [ ]:
# ── 4-3. SHAP 시각화 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar plot: 상위 SHAP_TOP_K feature
top_feats = shap_importance.head(SHAP_TOP_K)
colors = cm.viridis(np.linspace(0.2, 0.8, len(top_feats)))[::-1]
axes[0].barh(top_feats.index[::-1], top_feats.values[::-1], color=colors)
axes[0].set_title(f'SHAP Feature Importance (Top {SHAP_TOP_K})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('mean |SHAP value|')
axes[0].axvline(0, color='black', lw=0.5)

# Cumulative importance curve
cumsum_pct = shap_importance.cumsum() / shap_importance.sum() * 100
axes[1].plot(range(1, len(cumsum_pct) + 1), cumsum_pct.values, 'steelblue', lw=2)
axes[1].axhline(90, color='red', ls='--', lw=1, label='90%')
axes[1].axhline(95, color='orange', ls='--', lw=1, label='95%')
axes[1].axvline(SHAP_TOP_K, color='green', ls='--', lw=1.5, label=f'Top-{SHAP_TOP_K}')
axes[1].set_title('Cumulative SHAP Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Features'); axes[1].set_ylabel('Cumulative Importance (%)')
axes[1].legend(); axes[1].set_xlim(1, len(feat_cols_clean))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_shap_importance.png', bbox_inches='tight')
plt.show()

top_k_pct = cumsum_pct.iloc[SHAP_TOP_K - 1]
print(f'\nTop-{SHAP_TOP_K} feature → 전체 SHAP 중요도의 {top_k_pct:.1f}% 커버')

In [ ]:
# ── 4-4. SHAP Beeswarm plot ───────────────────────────────────
shap_exp = shap.Explanation(
    values=shap_values[:, :SHAP_TOP_K],
    base_values=explainer.expected_value,
    data=X_shap[:, :SHAP_TOP_K],
    feature_names=feat_cols_clean[:SHAP_TOP_K],
)
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp, max_display=SHAP_TOP_K, show=False)
plt.title(f'SHAP Beeswarm (Top {SHAP_TOP_K} features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_shap_beeswarm.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4-5. 선택된 Feature로 데이터 축소 ────────────────────────
selected_features = shap_importance.head(SHAP_TOP_K).index.tolist()

# 전체 데이터에서 선택된 feature만 추출
X_sel = X_scaled_df[selected_features].values

print(f'선택된 feature: {len(selected_features)}개')
print(f'클러스터링에 사용될 데이터 shape: {X_sel.shape}')
print(f'\n선택된 features:')
for i, f in enumerate(selected_features, 1):
    print(f'  {i:2d}. {f}  (SHAP={shap_importance[f]:.4f})')

## 5. Cost Function 정의

In [ ]:
def compute_cluster_sigmas(labels: np.ndarray, cd: np.ndarray) -> dict:
    """클러스터별 CD sigma(std) 계산.
    
    Returns
    -------
    dict with keys:
        sigma_per_cluster : {label: sigma}
        mean_sigma        : 클러스터 sigma의 평균 (샘플 수 가중)
        unweighted_mean   : 단순 평균
        max_sigma         : 클러스터 sigma 최대값
        cluster_counts    : {label: count}
        min_count         : 최소 클러스터 크기
    """
    unique_labels = np.unique(labels)
    sigma_per_cluster = {}
    counts = {}
    for lbl in unique_labels:
        mask = labels == lbl
        sigma_per_cluster[lbl] = cd[mask].std()
        counts[lbl] = mask.sum()

    sigmas = np.array(list(sigma_per_cluster.values()))
    ns     = np.array([counts[l] for l in unique_labels])

    return {
        'sigma_per_cluster': sigma_per_cluster,
        'mean_sigma'        : np.average(sigmas, weights=ns),  # 샘플 수 가중 평균
        'unweighted_mean'   : sigmas.mean(),                   # 단순 평균
        'max_sigma'         : sigmas.max(),
        'cluster_counts'    : counts,
        'min_count'         : ns.min(),
        'n_clusters'        : len(unique_labels),
    }


def cost_function(
    labels: np.ndarray,
    cd: np.ndarray,
    alpha: float = ALPHA,
    beta: float = BETA,
    min_count: int = MIN_CLUSTER_COUNT,
) -> float:
    """Cost = α × mean(σ) + β × max(σ).
    
    제약 위반 시 페널티(inf) 적용.
    """
    stats = compute_cluster_sigmas(labels, cd)
    if stats['min_count'] < min_count:
        return float('inf')  # 제약 위반 페널티
    return alpha * stats['mean_sigma'] + beta * stats['max_sigma']


def print_cluster_stats(stats: dict, method_name: str = '') -> None:
    """클러스터 통계 출력."""
    print(f'\n── {method_name} 클러스터 통계 ──')
    print(f'  클러스터 수          : {stats["n_clusters"]}')
    print(f'  최소 클러스터 크기   : {stats["min_count"]:,}')
    print(f'  가중 평균 σ (CD)     : {stats["mean_sigma"]:.4f} nm')
    print(f'  단순 평균 σ (CD)     : {stats["unweighted_mean"]:.4f} nm')
    print(f'  최대 σ (CD)          : {stats["max_sigma"]:.4f} nm')
    print(f'  Cost (α={ALPHA},β={BETA}): {ALPHA*stats["mean_sigma"]+BETA*stats["max_sigma"]:.4f}')

print('Cost function 정의 완료.')
print(f'  Cost = {ALPHA} × mean(σ) + {BETA} × max(σ)')
print(f'  제약: 클러스터 최소 {MIN_CLUSTER_COUNT}개 샘플')

## 6. Decision Tree Clustering (주 방법)

In [ ]:
# ── 6-1. 소규모 클러스터 병합 함수 ───────────────────────────
def merge_small_clusters(
    labels: np.ndarray,
    X: np.ndarray,
    min_count: int,
) -> np.ndarray:
    """크기가 min_count 미만인 클러스터를 KNN으로 가장 가까운 클러스터에 병합.

    반복적으로 수행하여 모든 클러스터가 제약을 만족할 때까지 진행.
    """
    labels = labels.copy()
    max_iter = 1000

    for _ in range(max_iter):
        counts = Counter(labels)
        small = [lbl for lbl, cnt in counts.items() if cnt < min_count]
        if not small:
            break

        # 가장 작은 클러스터부터 처리
        small_sorted = sorted(small, key=lambda l: counts[l])
        target_lbl = small_sorted[0]
        valid_lbls = [l for l in np.unique(labels) if l != target_lbl and counts[l] >= min_count]

        if not valid_lbls:
            # 유효 클러스터가 없으면 가장 큰 클러스터로 병합
            valid_lbls = [l for l in np.unique(labels) if l != target_lbl]
        if not valid_lbls:
            break

        # 소규모 클러스터 centroid와 가장 가까운 유효 클러스터 찾기
        small_mask = labels == target_lbl
        small_centroid = X[small_mask].mean(axis=0, keepdims=True)

        valid_centroids = np.array([
            X[labels == l].mean(axis=0) for l in valid_lbls
        ])
        dists = np.linalg.norm(valid_centroids - small_centroid, axis=1)
        nearest_lbl = valid_lbls[dists.argmin()]

        labels[small_mask] = nearest_lbl

    return labels


def relabel_sequential(labels: np.ndarray) -> np.ndarray:
    """클러스터 레이블을 0, 1, 2, ... 순서로 재할당."""
    unique = sorted(np.unique(labels))
    mapping = {old: new for new, old in enumerate(unique)}
    return np.vectorize(mapping.get)(labels)


print('병합 유틸리티 함수 정의 완료.')

In [ ]:
# ── 6-2. DT Grid Search: max_leaf_nodes 탐색 ─────────────────
print('Decision Tree 하이퍼파라미터 탐색 중...')
print(f'탐색 범위: max_leaf_nodes = {DT_MAX_LEAF_RANGE}')

dt_results = []

for max_leaves in tqdm(DT_MAX_LEAF_RANGE, desc='DT Grid Search'):
    dt = DecisionTreeRegressor(
        max_leaf_nodes=max_leaves,
        min_samples_leaf=MIN_CLUSTER_COUNT,  # 제약 1차 적용
        random_state=RANDOM_STATE,
    )
    dt.fit(X_sel, y)
    labels_raw = dt.apply(X_sel)  # 각 샘플의 leaf node ID

    # leaf node ID → 0-based sequential label
    labels_seq = relabel_sequential(labels_raw)

    # 병합 (혹시 남은 소규모 클러스터 처리)
    labels_merged = merge_small_clusters(labels_seq, X_sel, MIN_CLUSTER_COUNT)
    labels_final  = relabel_sequential(labels_merged)

    stats = compute_cluster_sigmas(labels_final, y)
    cost  = cost_function(labels_final, y)

    dt_results.append({
        'max_leaves'       : max_leaves,
        'n_clusters'       : stats['n_clusters'],
        'min_count'        : stats['min_count'],
        'mean_sigma'       : stats['mean_sigma'],
        'unweighted_mean'  : stats['unweighted_mean'],
        'max_sigma'        : stats['max_sigma'],
        'cost'             : cost,
        'labels'           : labels_final,
    })

dt_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'labels'} for r in dt_results])
print('\n탐색 결과 (상위 5):' )
print(dt_df.sort_values('cost').head())

In [ ]:
# ── 6-3. 최적 DT 설정 선택 ───────────────────────────────────
valid_dt = dt_df[dt_df['cost'] < float('inf')]
if valid_dt.empty:
    print('[경고] 제약 만족 DT 설정 없음 → min_leaves 완화 필요')
    best_dt_row = dt_df.sort_values('cost').iloc[0]
else:
    best_dt_row = valid_dt.sort_values('cost').iloc[0]

best_max_leaves = int(best_dt_row['max_leaves'])
best_dt_labels = dt_results[DT_MAX_LEAF_RANGE.index(best_max_leaves)]['labels']
best_dt_stats  = compute_cluster_sigmas(best_dt_labels, y)

print(f'최적 DT 설정: max_leaf_nodes = {best_max_leaves}')
print_cluster_stats(best_dt_stats, 'Decision Tree (최적)')

# Grid search 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(dt_df['max_leaves'], dt_df['mean_sigma'], 'o-', color='steelblue', label='Weighted Mean σ')
axes[0].plot(dt_df['max_leaves'], dt_df['unweighted_mean'], 's--', color='cornflowerblue', label='Unweighted Mean σ', alpha=0.7)
axes[0].axvline(best_max_leaves, color='red', ls='--', lw=1.5, label=f'Best ({best_max_leaves})')
axes[0].set_title('DT: Mean σ vs Max Leaves'); axes[0].set_xlabel('max_leaf_nodes'); axes[0].legend()

axes[1].plot(dt_df['max_leaves'], dt_df['max_sigma'], 'o-', color='salmon', label='Max σ')
axes[1].axvline(best_max_leaves, color='red', ls='--', lw=1.5)
axes[1].set_title('DT: Max σ vs Max Leaves'); axes[1].set_xlabel('max_leaf_nodes')

valid_cost = dt_df[dt_df['cost'] < float('inf')]
axes[2].plot(valid_cost['max_leaves'], valid_cost['cost'], 'o-', color='mediumseagreen', label='Cost')
axes[2].axvline(best_max_leaves, color='red', ls='--', lw=1.5, label=f'Best ({best_max_leaves})')
axes[2].set_title(f'DT: Cost (α={ALPHA}·mean_σ + β={BETA}·max_σ)'); axes[2].set_xlabel('max_leaf_nodes'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_dt_gridsearch.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6-4. 최적 DT 모델 재학습 & 저장 ──────────────────────────
best_dt = DecisionTreeRegressor(
    max_leaf_nodes=best_max_leaves,
    min_samples_leaf=MIN_CLUSTER_COUNT,
    random_state=RANDOM_STATE,
)
best_dt.fit(X_sel, y)
joblib.dump(best_dt, f'{OUTPUT_DIR}/best_dt_model.pkl')

# DT 규칙 텍스트 출력 (상위 수준)
print('Decision Tree 규칙 (depth=3까지):')
print(export_text(best_dt, feature_names=selected_features, max_depth=3))

## 7. K-Means Clustering (비교 기준선)

In [ ]:
# ── 7-1. K-Means Grid Search ──────────────────────────────────
print('K-Means 하이퍼파라미터 탐색 중...')
print(f'탐색 범위: k = {KMEANS_K_RANGE}')

km_results = []

for k in tqdm(KMEANS_K_RANGE, desc='KMeans Grid Search'):
    # MiniBatchKMeans: 280K 행에서도 빠름
    km = MiniBatchKMeans(
        n_clusters=k,
        batch_size=10_000,
        n_init=5,
        random_state=RANDOM_STATE,
    )
    labels_raw = km.fit_predict(X_sel)

    # 소규모 클러스터 병합
    labels_merged = merge_small_clusters(labels_raw, X_sel, MIN_CLUSTER_COUNT)
    labels_final  = relabel_sequential(labels_merged)

    stats = compute_cluster_sigmas(labels_final, y)
    cost  = cost_function(labels_final, y)

    km_results.append({
        'k'               : k,
        'n_clusters'      : stats['n_clusters'],
        'min_count'       : stats['min_count'],
        'mean_sigma'      : stats['mean_sigma'],
        'unweighted_mean' : stats['unweighted_mean'],
        'max_sigma'       : stats['max_sigma'],
        'cost'            : cost,
        'labels'          : labels_final,
    })

km_df = pd.DataFrame([{kk: v for kk, v in r.items() if kk != 'labels'} for r in km_results])
print('\n탐색 결과 (상위 5):')
print(km_df.sort_values('cost').head())

In [ ]:
# ── 7-2. 최적 K-Means ─────────────────────────────────────────
valid_km = km_df[km_df['cost'] < float('inf')]
if valid_km.empty:
    best_km_row = km_df.sort_values('cost').iloc[0]
else:
    best_km_row = valid_km.sort_values('cost').iloc[0]

best_k = int(best_km_row['k'])
best_km_labels = km_results[KMEANS_K_RANGE.index(best_k)]['labels']
best_km_stats  = compute_cluster_sigmas(best_km_labels, y)

print(f'최적 K-Means 설정: k = {best_k}')
print_cluster_stats(best_km_stats, 'K-Means (최적)')

# Grid search 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(km_df['k'], km_df['mean_sigma'], 'o-', color='steelblue', label='Weighted Mean σ')
axes[0].plot(km_df['k'], km_df['unweighted_mean'], 's--', color='cornflowerblue', label='Unweighted Mean σ', alpha=0.7)
axes[0].axvline(best_k, color='red', ls='--', lw=1.5, label=f'Best (k={best_k})')
axes[0].set_title('KMeans: Mean σ vs k'); axes[0].set_xlabel('k'); axes[0].legend()

axes[1].plot(km_df['k'], km_df['max_sigma'], 'o-', color='salmon')
axes[1].axvline(best_k, color='red', ls='--', lw=1.5)
axes[1].set_title('KMeans: Max σ vs k'); axes[1].set_xlabel('k')

valid_km_cost = km_df[km_df['cost'] < float('inf')]
axes[2].plot(valid_km_cost['k'], valid_km_cost['cost'], 'o-', color='mediumseagreen')
axes[2].axvline(best_k, color='red', ls='--', lw=1.5, label=f'Best (k={best_k})')
axes[2].set_title(f'KMeans: Cost'); axes[2].set_xlabel('k'); axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_kmeans_gridsearch.png', bbox_inches='tight')
plt.show()

## 8. 방법 비교 & 최종 선택

In [ ]:
# ── 8-1. 두 방법 비교 ─────────────────────────────────────────
dt_cost = cost_function(best_dt_labels, y)
km_cost = cost_function(best_km_labels, y)

comparison = pd.DataFrame([
    {
        'Method'         : 'Decision Tree',
        'n_clusters'     : best_dt_stats['n_clusters'],
        'min_count'      : best_dt_stats['min_count'],
        'mean_sigma (nm)': round(best_dt_stats['mean_sigma'], 4),
        'max_sigma (nm)' : round(best_dt_stats['max_sigma'], 4),
        'Cost'           : round(dt_cost, 4),
        'Interpretable'  : 'Yes (rule-based)',
    },
    {
        'Method'         : 'K-Means',
        'n_clusters'     : best_km_stats['n_clusters'],
        'min_count'      : best_km_stats['min_count'],
        'mean_sigma (nm)': round(best_km_stats['mean_sigma'], 4),
        'max_sigma (nm)' : round(best_km_stats['max_sigma'], 4),
        'Cost'           : round(km_cost, 4),
        'Interpretable'  : 'No (centroid)',
    },
])
print('방법 비교:')
display(comparison)

# 최종 선택 (cost 기준)
if dt_cost <= km_cost:
    FINAL_LABELS  = best_dt_labels
    FINAL_STATS   = best_dt_stats
    FINAL_METHOD  = 'Decision Tree'
    FINAL_COST    = dt_cost
else:
    FINAL_LABELS  = best_km_labels
    FINAL_STATS   = best_km_stats
    FINAL_METHOD  = 'K-Means'
    FINAL_COST    = km_cost

print(f'\n최종 선택: {FINAL_METHOD} (Cost={FINAL_COST:.4f})')

In [ ]:
# ── 8-2. 전체 sigma 대비 개선율 ──────────────────────────────
baseline_sigma = y.std()
improvement_mean = (baseline_sigma - FINAL_STATS['mean_sigma']) / baseline_sigma * 100
improvement_max  = (baseline_sigma - FINAL_STATS['max_sigma'])  / baseline_sigma * 100

print(f'\n[클러스터링 효과 요약]')
print(f'  전체 데이터 σ     : {baseline_sigma:.4f} nm')
print(f'  클러스터 평균 σ   : {FINAL_STATS["mean_sigma"]:.4f} nm  (개선율: {improvement_mean:.1f}%)')
print(f'  클러스터 최대 σ   : {FINAL_STATS["max_sigma"]:.4f} nm')
print(f'  클러스터 수        : {FINAL_STATS["n_clusters"]}')
print(f'  최소 클러스터 크기 : {FINAL_STATS["min_count"]:,}')

## 9. 결과 시각화

In [ ]:
# ── 9-1. 클러스터별 CD 분포 (Violin + Box) ───────────────────
n_cls = FINAL_STATS['n_clusters']
cluster_cd_data = [y[FINAL_LABELS == c] for c in range(n_cls)]
cluster_sizes   = [len(d) for d in cluster_cd_data]
cluster_sigmas  = [d.std() for d in cluster_cd_data]
cluster_means   = [d.mean() for d in cluster_cd_data]

# 클러스터를 CD 평균 기준으로 정렬
order = np.argsort(cluster_means)

fig, axes = plt.subplots(2, 1, figsize=(max(12, n_cls * 0.6), 12))

# Box plot
bp = axes[0].boxplot(
    [cluster_cd_data[i] for i in order],
    notch=True, patch_artist=True, showfliers=False,
)
colors_box = cm.RdYlGn_r(np.linspace(0.1, 0.9, n_cls))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_xticks(range(1, n_cls + 1))
axes[0].set_xticklabels([f'C{order[i]}\n(n={cluster_sizes[order[i]]:,})' for i in range(n_cls)],
                         fontsize=8, rotation=45, ha='right')
axes[0].set_title(f'{FINAL_METHOD} Clustering — CD (nm) per Cluster (sorted by mean)',
                   fontsize=13, fontweight='bold')
axes[0].set_ylabel('CD (nm)')
axes[0].axhline(y.mean(), color='navy', ls='--', lw=1, label='Overall Mean')
axes[0].legend()

# Sigma bar plot
sigma_ordered = [cluster_sigmas[i] for i in order]
bar_colors = cm.RdYlGn_r(np.array(sigma_ordered) / max(sigma_ordered))
bars = axes[1].bar(range(n_cls), sigma_ordered, color=bar_colors, edgecolor='white', lw=0.5)
axes[1].axhline(FINAL_STATS['mean_sigma'], color='blue', ls='--', lw=2,
                label=f'Weighted Mean σ = {FINAL_STATS["mean_sigma"]:.3f}')
axes[1].axhline(FINAL_STATS['max_sigma'], color='red', ls='-.', lw=2,
                label=f'Max σ = {FINAL_STATS["max_sigma"]:.3f}')
axes[1].axhline(baseline_sigma, color='gray', ls=':', lw=2,
                label=f'Baseline σ = {baseline_sigma:.3f}')
axes[1].set_xticks(range(n_cls))
axes[1].set_xticklabels([f'C{order[i]}' for i in range(n_cls)], fontsize=8, rotation=45, ha='right')
axes[1].set_title('σ per Cluster', fontsize=13, fontweight='bold')
axes[1].set_ylabel('σ of CD (nm)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_cluster_cd_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9-2. PCA 2D 시각화 ─────────────────────────────────────────
print('PCA 2D 투영 중...')
pca_sample_n = min(20_000, len(X_sel))
pca_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_sel), pca_sample_n, replace=False)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sel[pca_idx])
labels_pca = FINAL_LABELS[pca_idx]
cd_pca     = y[pca_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 클러스터 색상
palette = cm.tab20(np.linspace(0, 1, n_cls))
for c in range(n_cls):
    mask = labels_pca == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=3, alpha=0.4, color=palette[c], label=f'C{c}')
axes[0].set_title(f'PCA (2D) — {FINAL_METHOD} Clusters', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
if n_cls <= 20:
    axes[0].legend(markerscale=3, fontsize=8, ncol=2)

# CD 값으로 색칠
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=cd_pca,
                      cmap='plasma', s=3, alpha=0.5, vmin=np.percentile(cd_pca, 2),
                      vmax=np.percentile(cd_pca, 98))
plt.colorbar(sc, ax=axes[1], label='CD (nm)')
axes[1].set_title('PCA (2D) — CD Value', fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'PC1'); axes[1].set_ylabel(f'PC2')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_pca_visualization.png', bbox_inches='tight')
plt.show()
print(f'PCA 설명 분산: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
# ── 9-3. 클러스터 크기 & sigma 히트맵 ────────────────────────
cluster_summary = pd.DataFrame({
    'cluster'   : range(n_cls),
    'count'     : [cluster_sizes[i] for i in range(n_cls)],
    'mean_CD'   : [cluster_means[i] for i in range(n_cls)],
    'sigma_CD'  : [cluster_sigmas[i] for i in range(n_cls)],
    'pct'       : [cluster_sizes[i] / len(y) * 100 for i in range(n_cls)],
}).sort_values('sigma_CD')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 클러스터 크기
axes[0].bar(cluster_summary['cluster'].astype(str),
             cluster_summary['count'],
             color=cm.Blues(cluster_summary['count'] / cluster_summary['count'].max()))
axes[0].axhline(MIN_CLUSTER_COUNT, color='red', ls='--', lw=1.5, label=f'Min count={MIN_CLUSTER_COUNT}')
axes[0].set_title('Cluster Size Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Count'); axes[0].legend()
axes[0].tick_params(axis='x', labelsize=8)

# sigma vs count scatter
sc2 = axes[1].scatter(cluster_summary['count'], cluster_summary['sigma_CD'],
                       c=cluster_summary['mean_CD'], cmap='coolwarm', s=100, zorder=5, edgecolors='k', lw=0.5)
plt.colorbar(sc2, ax=axes[1], label='Mean CD (nm)')
axes[1].axhline(FINAL_STATS['mean_sigma'], color='blue', ls='--', lw=1.5, label='Mean σ')
axes[1].set_title('Cluster Size vs σ (colored by Mean CD)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster Size'); axes[1].set_ylabel('σ of CD (nm)'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_cluster_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9-4. 방법론 비교 시각화 ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['Baseline\n(All data)', 'Decision Tree', 'K-Means']
mean_sigmas = [baseline_sigma, best_dt_stats['mean_sigma'], best_km_stats['mean_sigma']]
max_sigmas  = [baseline_sigma, best_dt_stats['max_sigma'],  best_km_stats['max_sigma']]
costs_      = [ALPHA*baseline_sigma + BETA*baseline_sigma,
               dt_cost if dt_cost < float('inf') else 0,
               km_cost if km_cost < float('inf') else 0]

x = np.arange(len(methods))
w = 0.35
axes[0].bar(x - w/2, mean_sigmas, w, label='Mean σ', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, max_sigmas,  w, label='Max σ',  color='salmon',    alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(methods)
axes[0].set_title('Mean σ & Max σ Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('σ of CD (nm)'); axes[0].legend()

bar_colors2 = ['gray', 'mediumseagreen' if dt_cost <= km_cost else 'lightcoral',
               'mediumseagreen' if km_cost < dt_cost else 'lightcoral']
axes[1].bar(methods, costs_, color=bar_colors2, alpha=0.85, edgecolor='k', lw=0.5)
axes[1].set_title(f'Cost Comparison (α={ALPHA}·meanσ + β={BETA}·maxσ)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cost')
for i, (m, c) in enumerate(zip(methods, costs_)):
    axes[1].text(i, c + 0.01, f'{c:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_method_comparison.png', bbox_inches='tight')
plt.show()

## 10. SHAP 클러스터 해석 (Cluster-level SHAP)

In [ ]:
# ── 10-1. 클러스터별 feature 평균 히트맵 ─────────────────────
# 전체 데이터에서 서브샘플링하여 SHAP 기반 클러스터 특성 분석
heatmap_sample_n = min(5_000, len(X_sel))
hm_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_sel), heatmap_sample_n, replace=False)

hm_df = pd.DataFrame(X_sel[hm_idx], columns=selected_features)
hm_df['cluster'] = FINAL_LABELS[hm_idx]
hm_df['CD'] = y[hm_idx]

# 클러스터별 feature 평균 (z-score 정규화)
cluster_feat_means = hm_df.groupby('cluster')[selected_features].mean()
cluster_feat_std   = (cluster_feat_means - cluster_feat_means.mean()) / (cluster_feat_means.std() + 1e-8)

fig, ax = plt.subplots(figsize=(max(12, n_cls * 0.8), max(6, len(selected_features) * 0.35)))
sns.heatmap(
    cluster_feat_std.T,
    cmap='RdBu_r', center=0, fmt='.1f',
    linewidths=0.3, linecolor='lightgray',
    cbar_kws={'label': 'Z-score (feature mean per cluster)'},
    ax=ax,
)
ax.set_title('Cluster Feature Profile (Z-scored Mean, SHAP-selected Features)',
              fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_cluster_feature_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10-2. 클러스터별 CD 중앙값 & sigma 정리표 ────────────────
cluster_report = hm_df.groupby('cluster').agg(
    count=('CD', 'count'),
    CD_mean=('CD', 'mean'),
    CD_median=('CD', 'median'),
    CD_std=('CD', 'std'),
    CD_min=('CD', 'min'),
    CD_max=('CD', 'max'),
).round(3)
cluster_report['CD_range'] = (cluster_report['CD_max'] - cluster_report['CD_min']).round(3)
cluster_report = cluster_report.sort_values('CD_std')

print('클러스터별 CD 통계 요약 (sub-sample 기준):')
display(cluster_report)
cluster_report.to_csv(f'{OUTPUT_DIR}/cluster_report.csv')
print(f'\n저장: {OUTPUT_DIR}/cluster_report.csv')

## 11. 최종 결과 저장

In [ ]:
# ── 11-1. 전체 데이터에 클러스터 레이블 부여 ──────────────────
df_out = df_clean.copy()
df_out['cluster'] = FINAL_LABELS

# 클러스터별 sigma, mean 추가
cluster_cd_std  = df_out.groupby('cluster')[CD_COLUMN].std().rename('cluster_sigma')
cluster_cd_mean = df_out.groupby('cluster')[CD_COLUMN].mean().rename('cluster_mean_CD')
df_out = df_out.join(cluster_cd_std,  on='cluster')
df_out = df_out.join(cluster_cd_mean, on='cluster')

out_path = f'{OUTPUT_DIR}/clustered_data.parquet'
df_out.to_parquet(out_path, index=True)
print(f'클러스터 결과 저장: {out_path}')
print(f'  행 수: {len(df_out):,}')
print(f'  컬럼: cluster, cluster_sigma, cluster_mean_CD 추가됨')

In [ ]:
# ── 11-2. 최종 요약 JSON 저장 ─────────────────────────────────
summary = {
    'method'                 : FINAL_METHOD,
    'n_clusters'             : int(FINAL_STATS['n_clusters']),
    'min_cluster_count'      : int(FINAL_STATS['min_count']),
    'n_features_used'        : len(selected_features),
    'selected_features'      : selected_features,
    'cost_alpha'             : ALPHA,
    'cost_beta'              : BETA,
    'cost_value'             : round(FINAL_COST, 6),
    'baseline_sigma_nm'      : round(float(baseline_sigma), 6),
    'mean_cluster_sigma_nm'  : round(float(FINAL_STATS['mean_sigma']), 6),
    'max_cluster_sigma_nm'   : round(float(FINAL_STATS['max_sigma']), 6),
    'sigma_improvement_pct'  : round(float(improvement_mean), 2),
    'cluster_sizes'          : {int(k): int(v) for k, v in FINAL_STATS['cluster_counts'].items()},
    'cluster_sigmas'         : {int(k): round(float(v), 4)
                                for k, v in FINAL_STATS['sigma_per_cluster'].items()},
    'config': {
        'MIN_CLUSTER_COUNT': MIN_CLUSTER_COUNT,
        'SHAP_TOP_K'       : SHAP_TOP_K,
        'RANDOM_STATE'     : RANDOM_STATE,
    },
}
json_path = f'{OUTPUT_DIR}/clustering_summary.json'
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'요약 JSON 저장: {json_path}')

# SHAP 중요도 저장
shap_importance.to_csv(f'{OUTPUT_DIR}/shap_importance.csv', header=['mean_abs_shap'])
print(f'SHAP 중요도 저장: {OUTPUT_DIR}/shap_importance.csv')

In [ ]:
# ── 11-3. 최종 요약 출력 ──────────────────────────────────────
print('=' * 60)
print('  In-House Layout Clustering — 최종 결과')
print('=' * 60)
print(f'  선택 방법        : {FINAL_METHOD}')
print(f'  사용 features    : {len(selected_features)}개 (SHAP Top-{SHAP_TOP_K})')
print(f'  클러스터 수       : {FINAL_STATS["n_clusters"]}')
print(f'  최소 클러스터 크기: {FINAL_STATS["min_count"]:,} ≥ {MIN_CLUSTER_COUNT}')
print(f'  기준선 σ          : {baseline_sigma:.4f} nm')
print(f'  클러스터 평균 σ   : {FINAL_STATS["mean_sigma"]:.4f} nm ({improvement_mean:+.1f}%)')
print(f'  클러스터 최대 σ   : {FINAL_STATS["max_sigma"]:.4f} nm')
print(f'  Cost              : {FINAL_COST:.4f}')
print('=' * 60)
print(f'  출력 파일:')
for f in Path(OUTPUT_DIR).iterdir():
    print(f'    {f.name}')
print('=' * 60)